In [ ]:
# --- Deep learning ---
from torch.utils.data import DataLoader
import torch
from torchvision import transforms

# --- Data processing ---
import os
import numpy as np
import pandas as pd
from pathlib import Path

# --- Image I/O ---
from PIL import ImageFile, Image
Image.MAX_IMAGE_PIXELS = None
from skimage import io, img_as_float32
import matplotlib.pyplot as plt

# --- Whole-slide image ---
import openslide
from openslide import OpenSlide

# --- Progress bar ---
from tqdm import tqdm

# --- Spatial transcriptomics ---
import scanpy as sc
from sklearn.neighbors import NearestNeighbors

# --- Pretrained models ---
from transformers import AutoImageProcessor, AutoModel

# --- Custom modules ---
from finetune.vision_transformer import vit_small

## Dense Sliding Window

Samples every possible `patch_size × patch_size` window with stride = `patch_size`.
Filters white/blank patches using RGB mean thresholds.

In [ ]:
TCGA_img_dir = r'./实验/实验数据/TCGA-BRCA数据/Slide_Image/'
patch_size = 20
rgb_thresholds = {'r_threshold': 220, 'g_threshold': 220, 'b_threshold': 220}

for name in os.listdir(TCGA_img_dir):
    sample_dir = os.path.join(TCGA_img_dir, name)
    if not os.path.isdir(sample_dir):
        continue

    jpg_path = os.path.join(sample_dir, f'{name}.jpg')
    if not os.path.exists(jpg_path):
        print(f"SKIP {name}: JPEG not found")
        continue

    im = io.imread(jpg_path)
    im = img_as_float32(im)
    im = (255 * im).astype("uint8")
    height, width, channels = im.shape
    if channels == 4:
        im = im[:, :, :3]
    print(f"{name} image: {width} x {height}")

    num_patches_x = width // patch_size
    num_patches_y = height // patch_size
    print(f"{name} patches: {num_patches_x} x {num_patches_y} = {num_patches_x * num_patches_y}")

    spot_coordinates = []
    patch_count, saved_count = 0, 0

    for y_idx, y in enumerate(tqdm(range(0, num_patches_y * patch_size, patch_size),
                                     desc=f"Processing {name}")):
        for x_idx, x in enumerate(range(0, num_patches_x * patch_size, patch_size)):
            patch_count += 1
            patch = im[y:y+patch_size, x:x+patch_size, :]
            mean_rgb = np.mean(patch, axis=(0, 1))

            # Skip white/blank patches
            if (mean_rgb[0] > rgb_thresholds['r_threshold'] and
                mean_rgb[1] > rgb_thresholds['g_threshold'] and
                mean_rgb[2] > rgb_thresholds['b_threshold']):
                continue

            center_x = x + patch_size // 2
            center_y = y + patch_size // 2
            spot_coordinates.append({
                'array_row': x_idx,
                'array_col': y_idx,
                'x': center_x,
                'y': center_y
            })
            saved_count += 1

    print(f"OK {name}: {patch_count} total, {saved_count} kept")
    # TSV saving is commented out — use Method 2 output instead